In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate, correlation_lags
from pathlib import Path
import napari

# ==========================================
# 1. CONSENSUS-BASED TFM STITCHER CLASS
# ==========================================
class TFMStitcher:
    def __init__(self, vol1, vol2, axis=2):
        self.vol1 = vol1.astype(np.float32)
        self.vol2 = vol2.astype(np.float32)
        self.axis = axis
        self.shift_index = 0
        self.best_score = -1

    def find_optimal_shift(self, grid=(120, 40), ignore_top=30, expected=0, tolerance=200, cutoff_db=-10):
        v1_abs, v2_abs = np.abs(self.vol1), np.abs(self.vol2)
        global_max = max(np.max(v1_abs), np.max(v2_abs))
        thresh = global_max * (10**(cutoff_db / 20))
        
        v1_t = np.where(v1_abs >= thresh, self.vol1, 0)
        v2_t = np.where(v2_abs >= thresh, self.vol2, 0)

        z_dim = v1_t.shape[0]
        tile_z = (z_dim - ignore_top) // grid[0]
        tile_y = v1_t.shape[1] // grid[1]
        
        all_shifts, all_weights = [], []
        
        for r in range(grid[0]):
            for c in range(grid[1]):
                zs, ze = ignore_top + (r * tile_z), ignore_top + ((r + 1) * tile_z)
                ys, ye = c * tile_y, (c + 1) * tile_y
                
                prof1 = np.max(np.abs(v1_t[zs:ze, ys:ye, :]), axis=(0, 1))
                prof2 = np.max(np.abs(v2_t[zs:ze, ys:ye, :]), axis=(0, 1))

                if np.max(prof1) == 0 or np.max(prof2) == 0: continue
                
                p1_n = (prof1 - np.mean(prof1)) / (np.std(prof1) + 1e-10)
                p2_n = (prof2 - np.mean(prof2)) / (np.std(prof2) + 1e-10)
                
                corr = correlate(p1_n, p2_n, mode='full')
                lags = correlation_lags(len(p1_n), len(p2_n), mode='full')
                
                mask = (lags >= expected - tolerance) & (lags <= expected + tolerance)
                mask = mask & (lags >= 0) 
                if not np.any(mask): continue
                corr[~mask] = -np.inf 

                peak_idx = np.argmax(corr)
                bias = 0.5 + (0.5 * (c / grid[1])) 
                
                all_shifts.append(lags[peak_idx])
                all_weights.append(corr[peak_idx] * bias)

        if not all_shifts:
            self.shift_index = 0
            return 0

        bins = np.arange(np.min(all_shifts), np.max(all_shifts) + 2) - 0.5
        counts, bin_edges = np.histogram(all_shifts, bins=bins, weights=all_weights)
        self.shift_index = int(bin_edges[np.argmax(counts)] + 0.5)
        self.best_score = np.max(counts)
        return self.shift_index

    def stitch(self, blend_mode='linear'):
        shift = int(round(self.shift_index))
        s1, s2 = self.vol1.shape, self.vol2.shape
        L1, L2 = s1[self.axis], s2[self.axis]
        
        v1_start, v2_start = (abs(shift), 0) if shift < 0 else (0, shift)
        total_len = max(v1_start + L1, v2_start + L2)
        stitched = np.zeros((*s1[:self.axis], total_len, *s1[self.axis+1:]), dtype=np.float32)
        
        sl1, sl2 = [slice(None)]*3, [slice(None)]*3
        sl1[self.axis], sl2[self.axis] = slice(v1_start, v1_start+L1), slice(v2_start, v2_start+L2)
        stitched[tuple(sl1)] = self.vol1
        
        inter_s, inter_e = max(v1_start, v2_start), min(v1_start + L1, v2_start + L2)
        if inter_e > inter_s:
            overlap = inter_e - inter_s
            ramp_shape = [1, 1, 1]; ramp_shape[self.axis] = overlap
            ramp = np.linspace(0, 1, overlap).reshape(ramp_shape)
            if shift < 0: ramp = 1.0 - ramp
            
            idx1, idx2 = [slice(None)]*3, [slice(None)]*3
            idx1[self.axis], idx2[self.axis] = slice(inter_s-v1_start, inter_e-v1_start), slice(inter_s-v2_start, inter_e-v2_start)
            stitched[tuple(idx1)] = (self.vol1[tuple(idx1)] * (1-ramp)) + (self.vol2[tuple(idx2)] * ramp)

        if v2_start + L2 > inter_e:
            end_sl, src_sl = [slice(None)]*3, [slice(None)]*3
            end_sl[self.axis] = slice(inter_e, v2_start+L2)
            src_sl[self.axis] = slice(inter_e-v2_start, L2)
            stitched[tuple(end_sl)] = self.vol2[tuple(src_sl)]
        return stitched

# ==========================================
# 2. MAIN SEQUENTIAL EXECUTION
# ==========================================
if __name__ == "__main__":
    DATA_ROOT = Path.cwd().parent / 'DATA' / '2D TFM Data'
    FILT_DIR = DATA_ROOT / 'FeC Smile 3MHz 04022026 Filtered'
    RAW_DIR = DATA_ROOT / 'FeC Smile 3MHz 04022026'

    file_ids = [9, 8, 7, 6, 5, 4, 3, 2, 1]
    
    print("Starting Hybrid Tail-Slicing Cross-Dimensional Stitching...")
    
    # Initialize both assemblies
    # We keep a filtered one for calculation and a raw one for the final image
    global_filt = np.load(FILT_DIR / f"FeC_40_{file_ids[0]}_filtered_3D_TFM.npy")
    global_raw = np.load(RAW_DIR / f"FeC_40_{file_ids[0]}_3D_TFM.npy")

    for i in range(1, len(file_ids)):
        fid = file_ids[i]
        print(f"\nMerging {i+1}/{len(file_ids)}: ID {fid}")

        # 1. Load the next volumes
        next_filt = np.load(FILT_DIR / f"FeC_40_{fid}_filtered_3D_TFM.npy")
        next_raw = np.load(RAW_DIR / f"FeC_40_{fid}_3D_TFM.npy")

        # 2. Check dimensions
        if next_filt.shape != next_raw.shape:
            raise ValueError(f"Shape mismatch in ID {fid}!")
        

        # 3. TAIL-SLICING (Logic from Code 1)
        # We find the shift by comparing the end of the FILTERED assembly to the NEXT filtered volume
        window_size = int(next_filt.shape[2] * 1.2)
        offset_removed = max(0, global_filt.shape[2] - window_size)
        active_tail_filt = global_filt[:, :, offset_removed:]

        # 4. FIND SHIFT (Using Filtered Tail)
        stitcher_calc = TFMStitcher(active_tail_filt, next_filt, axis=2)
        relative_shift = stitcher_calc.find_optimal_shift(grid=(120, 40), cutoff_db=-5)
        
        # Calculate Global Shift
        global_shift = offset_removed + relative_shift
        print(f" -> Derived Shift: {relative_shift} px | Global Pos: {global_shift} px")

        # 5. STITCH BOTH ASSEMBLIES
        # Update Filtered Assembly (to keep the 'search tail' updated)
        filt_stitcher = TFMStitcher(global_filt, next_filt, axis=2)
        filt_stitcher.shift_index = global_shift
        global_filt = filt_stitcher.stitch()

        # Update Raw Assembly (the actual output)
        raw_stitcher = TFMStitcher(global_raw, next_raw, axis=2)
        raw_stitcher.shift_index = global_shift
        global_raw = raw_stitcher.stitch()
        
    # --- VISUALIZATION ---
    v_min, v_max = np.percentile(global_raw, [0.1, 99.9])
    clim = sorted([float(v_min), float(v_max)])

    viewer = napari.Viewer(title="Hybrid Sequential Cross-Dimensional Assembly")
    viewer.add_image(global_raw, name="Final Assembly (Raw)", contrast_limits=clim, colormap="viridis")
    
    # Optional: Toggle this to see what the algorithm was 'looking' at
    viewer.add_image(global_filt, name="Filtered Assembly (Math Reference)", visible=False)
    
    napari.run()

Starting Hybrid Tail-Slicing Cross-Dimensional Stitching...

Merging 2/9: ID 8
 -> Derived Shift: 107 px | Global Pos: 107 px

Merging 3/9: ID 7
 -> Derived Shift: 130 px | Global Pos: 197 px

Merging 4/9: ID 6
 -> Derived Shift: 139 px | Global Pos: 296 px

Merging 5/9: ID 5
 -> Derived Shift: 95 px | Global Pos: 351 px

Merging 6/9: ID 4
 -> Derived Shift: 157 px | Global Pos: 468 px

Merging 7/9: ID 3
 -> Derived Shift: 161 px | Global Pos: 589 px

Merging 8/9: ID 2
 -> Derived Shift: 155 px | Global Pos: 704 px

Merging 9/9: ID 1
 -> Derived Shift: 133 px | Global Pos: 797 px
